# R8 · Cross-dataset partial queries (Fig 4A)

Four breast gene programmes are searched as partial queries in other sections:
* **Exemplar per programme:** the breast seed-0 index tile with the largest robust co-variation on the programme genes.
  R_S is its correlation and q_g its variance quantiles.
* **Query:** diag(σ) R_S diag(σ), with σ quantile-matched to the section searched.
* **Skin, seed 0:** every index tile's exact d_B^S to each query, for the closeness maps.

The rules are fixed in `experiments/common/pan_cancer.py`; R9 adds the cells, the tiles Spindle returns and every other
section.

* **Prerequisites:** `R0_index_cache.ipynb`.
* **Writes:** `results/pan_cancer/`: `exemplar.{csv,npz}`, `per_dataset/skin_melanoma/seed0_gradient.npz`.
* **Skips** existing outputs (`FORCE = True` redoes all).
* **Runtime / memory:** a few minutes on one core.

In [1]:
import gc
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd

import data
import pan_cancer as pc
import results_io as rio
import sections as sx

FORCE = False
TOPIC = "pan_cancer"
OUT = rio.topic(TOPIC)
print(data.stamp())


def save_npz(arrays, file, description):
    path = OUT / file
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)
    rio.register(TOPIC, path, description)

{'tag': 'method-v2', 'describe': 'method-v3-dirty', 'commit': 'b35f7d4'}


## Exemplars (breast, seed 0)

In [2]:
if FORCE or not (OUT / "exemplar.csv").exists() or not (OUT / "exemplar.npz").exists():
    E, arrays = pc.exemplars()
    rio.write_csv(E, TOPIC, "exemplar.csv", "one exemplar tile per programme: genes, c, mean r, variance quantiles q")
    save_npz(arrays, "exemplar.npz", "R_S, q and box of each exemplar; breast cells")
    del arrays
    gc.collect()
E = pd.read_csv(OUT / "exemplar.csv")
Z = dict(np.load(OUT / "exemplar.npz"))
E

,program,genes,n_genes,tile,n_cells,c,mean_r,n_robust,q,tag
0,Cytotoxic_T,CD3D CD3E CD8A NKG7 GNLY GZMA GZMB PRF1 KLRD1 ...,10,1120,179,0.403952,0.120997,191,0.803 0.702 0.829 0.866 0.786 0.527 0.734 0.87...,method-v2
1,Myeloid,CD68 CD163 CD14 AIF1 MNDA,5,832,180,0.654132,0.412213,270,0.791 0.692 0.829 0.805 0.743,method-v2
2,Stromal_Perivascular,ACTA2 MYH11 PDGFRB PDGFRA FBLN1,5,947,90,0.511000,0.283819,355,0.187 0.105 0.755 0.860 0.611,method-v2
3,B_Plasma,MS4A1 CD79A MZB1 TNFRSF17 DERL3,5,389,150,0.406669,0.174434,112,0.765 0.960 0.970 0.949 0.951,method-v2


## Skin, seed 0: closeness of every index tile to each query

In [3]:
f = f"per_dataset/{pc.GRADIENT_KEY}/seed{data.EXAMPLE_SEED}_gradient.npz"
if FORCE or not (OUT / f).exists():
    S = sx.load(pc.GRADIENT_KEY, data.EXAMPLE_SEED, parts=("layout", "naive_partial"))
    save_npz(pc.gradient(S, E, Z), f, "seed 0: cells, index tile centres/sides, exact d_B^S of every index tile to each query")
    del S
    gc.collect()
else:
    print(f"{f}: exists, skipped")

per_dataset/skin_melanoma/seed0_gradient.npz: exists, skipped


In [4]:
rio.write_manifest(TOPIC, "R8_pan_cancer.ipynb", inputs=["results/_cache"], notes="Fig 4A.")

wrote results/pan_cancer/manifest.json: 3 files; peak memory 0.16 GB


{'topic': 'pan_cancer',
 'notebook': 'R8_pan_cancer.ipynb',
 'date': '2026-11-05 09:40',
 'tag': 'method-v2',
 'describe': 'method-v3-dirty',
 'commit': 'b35f7d4',
 'inputs': ['results/_cache'],
 'peak_rss_gb': 0.16,
 'notes': 'Fig 4A.',
 'files': {'exemplar.csv': {'description': 'one exemplar tile per programme: genes, c, mean r, variance quantiles q',
   'rows': 4,
   'bytes': 655},
  'exemplar.npz': {'description': 'R_S, q and box of each exemplar; breast cells',
   'rows': None,
   'bytes': 1104544},
  'per_dataset/skin_melanoma/seed0_gradient.npz': {'description': 'seed 0: cells, index tile centres/sides, exact d_B^S of every index tile to each query',
   'rows': None,
   'bytes': 634605}}}